In [ ]:
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Cluster"
feature_cols = ["x1", "x2", "x3"]
df = pd.read_excel(file_path, sheet_name=sheet_name)
X = df[feature_cols].to_numpy(dtype=float)

# ========= 2) 参数模板 =========
params = {
    "n_clusters": 3,      # 聚类个数
    "linkage": "ward",    # 层次聚类连接方式
    "som_x": 5, "som_y": 5, "sigma": 1.0, "learning_rate": 0.5  # SOM参数
}

kmeans_label = KMeans(n_clusters=params["n_clusters"], random_state=42, n_init=10).fit_predict(X)
hier_label = AgglomerativeClustering(n_clusters=params["n_clusters"], linkage=params["linkage"]).fit_predict(X)
gmm_label = GaussianMixture(n_components=params["n_clusters"], random_state=42).fit_predict(X)

print("kmeans:", kmeans_label[:10])
print("hier:", hier_label[:10])
print("gmm:", gmm_label[:10])

# SOM（可选）
from minisom import MiniSom
som = MiniSom(params["som_x"], params["som_y"], X.shape[1], sigma=params["sigma"], learning_rate=params["learning_rate"])
som.random_weights_init(X)
som.train_random(X, 1000)
som_label = [som.winner(x)[0] * params["som_y"] + som.winner(x)[1] for x in X]
print("som:", som_label[:10])


In [ ]:
"""
k-means、层次聚类、高斯混合聚类、SOM

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "k-means、层次聚类、高斯混合聚类、SOM.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
FEATURE_COLUMNS = ["特征1", "特征2"]  # TODO: 请填写[聚类特征列名列表]，说明：数值型特征。
CLUSTER_COUNT = 3  # TODO: 请填写[聚类数]，说明：正整数，可结合轮廓系数选择。
SOM_X = 2  # TODO: 请填写[SOM 网格行数]，说明：正整数。
SOM_Y = 2  # TODO: 请填写[SOM 网格列数]，说明：正整数。
SOM_ITERATIONS = 500  # TODO: 请填写[SOM 训练次数]，说明：样本多时可增大。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    X = StandardScaler().fit_transform(data[FEATURE_COLUMNS].astype(float))
    result = data.copy()
    result["kmeans"] = KMeans(n_clusters=CLUSTER_COUNT, random_state=RANDOM_STATE, n_init="auto").fit_predict(X)
    result["层次聚类"] = AgglomerativeClustering(n_clusters=CLUSTER_COUNT).fit_predict(X)
    result["高斯混合"] = GaussianMixture(n_components=CLUSTER_COUNT, random_state=RANDOM_STATE).fit_predict(X)
    # SOM 需要 minisom: pip install minisom；未安装时跳过，避免影响其他聚类结果。
    try:
        from minisom import MiniSom
        som = MiniSom(SOM_X, SOM_Y, X.shape[1], sigma=1.0, learning_rate=0.5, random_seed=RANDOM_STATE)
        som.train_random(X, SOM_ITERATIONS)
        result["SOM"] = [som.winner(row)[0] * SOM_Y + som.winner(row)[1] for row in X]
    except Exception as exc:
        print("SOM 未运行：", exc)
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result.head())


if __name__ == "__main__":
    df = load_data()
    run_model(df)


# k-means、层次聚类、高斯混合聚类、SOM

## 输入说明

- 数据文件：默认读取脚本同目录下的 `data.csv`，也可以在代码顶部把 `DATA_FILE` 改为 `.xlsx` 或绝对路径。
- 数据格式：一般要求“一行一个样本/时刻/方案，一列一个变量/指标”。具体列名需要在代码顶部的 `TODO` 参数区填写。
- 示例：若模型需要特征 `特征1、特征2` 和目标列 `y`，表格可整理为：

| 特征1 | 特征2 | y |
|---:|---:|---:|
| 1.2 | 3.4 | 8.1 |
| 2.0 | 2.8 | 9.0 |

## 输出说明

- 控制台会打印核心结果，例如模型参数、评价指标、最优解、排名或预测值。
- 默认结果保存到代码顶部 `OUTPUT_FILE` 指定的文件。
- 若模型包含图形分析，会额外输出图片文件，例如箱型图 `boxplot.png`。

## 原理通俗解释

聚类模型 的核心思想是：先把实际问题抽象成可计算的数据结构，再用对应的数学规则寻找“预测值、分类结果、综合得分或最优方案”。代码中已经保留主要计算流程，比赛时重点是把题目数据整理成表格，并把 TODO 参数替换为题目含义一致的列名和约束。

## 适用场景

无标签样本分组、画像和结构探索。

## 局限性

聚类数选择主观，距离尺度会显著影响结果。

## 使用提示

- 运行前先检查缺失值、异常值和量纲；很多模型对数据尺度敏感。
- 所有 `TODO` 都应结合题目背景填写，不要直接使用示例列名。
- 建模论文中建议同时写明参数来源，例如权重来自 AHP/熵权法，预测步数来自题目要求。
